# Lab | Sequence Modeling with LSTM

In [ ]:
!wget -nc http://nlp.stanford.edu/data/glove.6B.zip
!unzip -n -q glove.6B.zip

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
# Load the dataset
df = pd.read_csv('dataset.csv')  # Make sure your CSV file is in the correct path
text_col = 'Text'
label_col = 'Category'

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1490, 3)


,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


# =============================================================================
# EXERCISE 1: TEXT PREPROCESSING
# =============================================================================

In [ ]:

def clean_text(text):
    """
    Clean and preprocess text data
    1. Convert text to lowercase
    2. Remove all characters except letters and spaces using regex
    3. Return the cleaned text
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove non-alphabetic characters (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    
    return text

# Apply text cleaning
df[text_col] = df[text_col].astype(str).apply(clean_text)

print("Sample cleaned texts:")
print(df[text_col].head(3).tolist())

# =============================================================================
# EXERCISE 2: LABEL ENCODING
# =============================================================================


In [ ]:

# Initialize LabelEncoder and encode the labels
encoder = LabelEncoder()
df[label_col] = encoder.fit_transform(df[label_col])

num_classes = len(encoder.classes_)
print(f"Number of classes: {num_classes}")
print(f"Classes: {encoder.classes_}")



# =============================================================================
# EXERCISE 3: TEXT TOKENIZATION AND PADDING
# =============================================================================


In [ ]:

# Initialize tokenizer with out-of-vocabulary token
tokenizer = Tokenizer(oov_token='<OOV>')

# Fit tokenizer on texts
tokenizer.fit_on_texts(df[text_col])

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print(f"Vocabulary size: {vocab_size}")

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(df[text_col])

# Pad sequences to max_len=100
max_len = 100
padded = pad_sequences(sequences, maxlen=max_len, padding='post')

print(f"Padded sequences shape: {padded.shape}")



# =============================================================================
# EXERCISE 4: TRAIN-VALIDATION-TEST SPLIT
# =============================================================================


In [ ]:
# Split data into train (70%), validation (15%), and test (15%)
labels = df[label_col].values

X_train, X_temp, y_train, y_temp = train_test_split(
    padded, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

# =============================================================================
# EXERCISE 5: LOAD PRE-TRAINED EMBEDDINGS (OPTIONAL)
# ============================================================================


In [ ]:
# Load GloVe embeddings and build an embedding matrix aligned with our word_index
embedding_dim = 100
embeddings_index = {}

with open('glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

print(f"Found {len(embeddings_index)} word vectors in GloVe.")

embedding_matrix = np.zeros((vocab_size, embedding_dim))
hits = 0
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        hits += 1

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Words covered by GloVe: {hits}/{len(word_index)} ({hits / len(word_index):.1%})")

# =============================================================================
# EXERCISE 6: BUILD MODEL FUNCTION
# =============================================================================


In [ ]:
# Build RNN or LSTM model
def build_model(rnn_type='LSTM', units=64):
    """Builds a text-classification model using a pretrained GloVe embedding
    followed by either a SimpleRNN or an LSTM layer."""
    model = Sequential()
    model.add(Input(shape=(max_len,)))
    # Pretrained GloVe weights, frozen (trainable=False) so we don't overwrite them
    # with only ~1500 training examples.
    model.add(Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False))

    if rnn_type == 'LSTM':
        model.add(LSTM(units))
    elif rnn_type == 'SimpleRNN':
        model.add(SimpleRNN(units))
    else:
        raise ValueError("rnn_type must be 'LSTM' or 'SimpleRNN'")

    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

# =============================================================================
# EXERCISE 7: MODEL TRAINING AND EVALUATION
# =============================================================================


In [ ]:
# Train and evaluate both a SimpleRNN and an LSTM model, so we can compare them
results = {}
histories = {}
models = {}

for rnn_type in ['SimpleRNN', 'LSTM']:
    print(f"\n{'=' * 60}\nTraining {rnn_type} model\n{'=' * 60}")

    model = build_model(rnn_type=rnn_type)
    model.summary()

    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1,
    )

    # Make predictions on test set
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Generate classification report
    print(f"\nClassification Report ({rnn_type}):")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))

    # Generate confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    models[rnn_type] = model
    histories[rnn_type] = history
    results[rnn_type] = {
        'confusion_matrix': cm,
        'test_accuracy': float((y_pred == y_test).mean()),
    }


# =============================================================================
# EXERCISE 8: VISUALIZATION
# =============================================================================


In [ ]:

# Plot training accuracy and loss for both models, side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, rnn_type in enumerate(['SimpleRNN', 'LSTM']):
    h = histories[rnn_type].history

    axes[0, i].plot(h['accuracy'], label='train')
    axes[0, i].plot(h['val_accuracy'], label='val')
    axes[0, i].set_title(f'{rnn_type} - Accuracy')
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].legend()

    axes[1, i].plot(h['loss'], label='train')
    axes[1, i].plot(h['val_loss'], label='val')
    axes[1, i].set_title(f'{rnn_type} - Loss')
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].legend()

plt.tight_layout()
plt.show()

# Plot confusion matrices for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i, rnn_type in enumerate(['SimpleRNN', 'LSTM']):
    sns.heatmap(
        results[rnn_type]['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
        xticklabels=encoder.classes_, yticklabels=encoder.classes_, ax=axes[i],
    )
    axes[i].set_title(f'{rnn_type} - Confusion Matrix')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')

plt.tight_layout()
plt.show()


# =============================================================================
# EXERCISE 9: RESULTS ANALYSIS
# =============================================================================


In [ ]:
# Add best validation metrics to results
for rnn_type in ['SimpleRNN', 'LSTM']:
    h = histories[rnn_type].history
    results[rnn_type]['best_val_accuracy'] = float(max(h['val_accuracy']))
    results[rnn_type]['best_val_loss'] = float(min(h['val_loss']))
    results[rnn_type]['epochs_trained'] = len(h['loss'])

results_df = pd.DataFrame({
    rnn_type: {
        'Best Val Accuracy': results[rnn_type]['best_val_accuracy'],
        'Best Val Loss': results[rnn_type]['best_val_loss'],
        'Test Accuracy': results[rnn_type]['test_accuracy'],
        'Epochs Trained (early stopping)': results[rnn_type]['epochs_trained'],
    }
    for rnn_type in ['SimpleRNN', 'LSTM']
}).T

print("Model comparison:")
print(results_df)

EXERCISE COMPLETE!